In [ ]:
import interfere
import interfere_experiments as ie
from interfere_experiments.data_generators import *
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np


DATA_DIR = "/Users/djpassey/Google Drive/My Drive/Docs/PhD Research/DissertationColabs/InterfereBenchmark1.1.1/"

BENCHMARK_MODELS = [
    AttractingFixedPoint4D,

    Belozyorov1,
    Belozyorov2,
    Belozyorov3,
    
    CoupledLogisticMapAllToAll,
    CoupledLogisticMapTwoCycles,
    CoupledLogisticMapOneCycle,
    CoupledLogisticMapConfound,
    CoupledLogisticMapBlockableConfound,

    CoupledMapLatticeChaoticBrownian,
    CoupledMapLatticeDefectTurbulence,
    CoupledMapLatticePatternSelection,
    CoupledMapLatticeSpatioTempChaos,
    CoupledMapLatticeSpatioTempIntermit,

    ImaginaryRoots4D,
    
    KuramotoOscilator1,
    KuramotoOscilator2,
    KuramotoOscilator3,
    KuramotoOscilator4,
    
    KuramotoSakaguchi1,
    KuramotoSakaguchi2,
    KuramotoSakaguchi3,

    Liping3DQuadFinance1,
    Liping3DQuadFinance2,

    Lorenz1,
    Lorenz2,

    LotkaVoltera1,
    LotkaVoltera2,
    LotkaVoltera3,
    LotkaVoltera4,
    LotkaVolteraConfound,
    LotkaVolteraMediator,

    LotkaVolteraBlockableConfound,

    MichaelisMenten1,
    MichaelisMenten2,
    MichaelisMenten3,

    MooreSpiegel1,
    MooreSpiegel2,
    MooreSpiegel3,

    MutualisticPop1,
    MutualisticPop1,
    MutualisticPop1,
    
    OrnsteinUhlenbeck1,
    OrnsteinUhlenbeck2,
    OrnsteinUhlenbeck3,

    PlantedTank1,
    PlantedTank2,
    PlantedTank3,

    Rossler1,
    Rossler2,

    SIS1,
    SIS2,
    SIS3,

    VARMA1SpatiotempChaos,
    VARMA2ChaoticBrownian,
    VARMA3LotkaVoltera, 

    Thomas1,
    Thomas2,

    WilsonCowanDownstream,
    WilsonCowanMultiConf,

    # Slow Models
    CoupledMapLatticeTravelingWave,
    CoupledMapLatticeChaoticTravelingWave,
]


COLORS = [str(v) for v in mcolors.TABLEAU_COLORS.values()]
plt.rcParams["figure.dpi"] = 150

TRAIN_OBS = 5000
FORECAST_OBS = 300
NUM_BURN_IN_STATES = 1000
INTERFERE_COMMIT_HASH = "b1c248a60bcca1781bceb9d5e60a9619a1c01ed5"


In [ ]:
def plot_helper(
    model_name: str,
    data: ie.control_vs_resp.ControlVsRespData,
    target_idx=None
):
    """Plots control vs Response Data"""

    dims = data.train_states.shape[1]
    all_states = np.vstack([
        data.train_states,
        data.forecast_states,
        data.interv_states,
    ])
    ymax = np.max(all_states)
    ymin = np.min(all_states)
    diff = ymax - ymin
    lims = (ymin - 0.1 * diff, ymax + 0.1 * diff)

    _, ax = plt.subplots(dims, 2, figsize=(7, 2 * dims))
    for i in range(dims):
        ax[i, 0].plot(
            data.train_t, data.train_states[:, i], alpha=0.7, c=COLORS[4])
        ax[i, 0].set_ylim(lims)

    for i in range(dims):
        ax[i, 1].plot(
            data.forecast_t, data.forecast_states[:, i], alpha=0.7, c=COLORS[0])
        if i == target_idx:
            ax[i, 1].plot(
            data.forecast_t, data.interv_states[:, i], alpha=0.7, lw=3, c=COLORS[6])
        else:
            ax[i, 1].plot(
                data.forecast_t, data.interv_states[:, i], alpha=0.7, c=COLORS[3])
        ax[i, 1].set_ylim(lims)

    ax[-1, -1].plot(
        data.forecast_t[:2], data.forecast_states[:2, -1], 
        alpha=0.7, c=COLORS[0], label="Forecast Test Data"
    )
    ax[-1, -1].plot(
        data.forecast_t[:2], data.interv_states[:2, -1], 
        alpha=0.7, c=COLORS[3], label="Intervention Test Data"
    )
    ax[-1, -1].plot(
        data.train_t[-2:], data.train_states[-2:, -1], 
        alpha=0.7, c=COLORS[4], label="Train Data"
    )

    plt.legend(loc=(0.2, -.5))
    plt.suptitle(model_name + "\nTraining Data (left) v.s. Forecast and Intervention Test Data (right)\n")
    plt.tight_layout()
    plt.show()

    # Error summary
    metric = interfere.metrics.rmse
    resp_target = data.interv_states[:, target_idx]
    fcast_target = data.forecast_states[:, target_idx]
    mu_pred = np.mean(data.train_states[:, target_idx])
    avg_pred =  mu_pred * np.ones_like(resp_target)

    fcast_mean_error = metric(fcast_target, avg_pred)
    resp_mean_error = metric(avg_pred, resp_target)


    print(model_name + "\nError Summary -- Predict Mean of Training Data:"
            f"\n\tForecast  {metric.__name__}: {fcast_mean_error}"
            f"\n\tResponse  {metric.__name__}: {resp_mean_error}"
            f"\n\tRatio re/fc: {resp_mean_error / fcast_mean_error}"  
    )
    return fcast_mean_error, resp_mean_error


def make_description(dg: ie.data_generators.DataGenerator):
    dg_name = f"DATA GENERATOR NAME: \n\t{type(dg).__name__}\n\n"
    model_type = f"MODEL_TYPE: \n\tinterfere.dynamics.{dg.model_type.__name__}\n\n"

    if isinstance(dg.model_type, type(interfere.DynamicModel)):
        init_descr = (
            f"DOCUMENTATION FOR MODEL: \n\n{dg.model_type.__init__.__doc__}\n\n"
        )
    else:
        init_descr = f"DOCUMENTATION FOR MODEL: \n\n{dg.model_type.__doc__}\n\n"

    param_str = "PARAMS:\n\n" + "\n".join(
        [
            f"{k} = {v}\n"
            for k, v in dg.model_params.items()
        ]
    )

    commit = f"\n\nINTERFERE COMMIT:\n\npip install git+https://github.com/djpasseyjr/interfere.git@{INTERFERE_COMMIT_HASH}"
    
    description = dg_name + model_type + init_descr + param_str + commit
    return description

In [ ]:
data = ie.control_vs_resp.load_cvr_json(DATA_DIR + "Deterministic/AttractingFixedPoint4d.json")

In [ ]:
sub_folders = ["Deterministic/", "LowStochastic/", "HighStochastic/"]
sigma_frac = [None, 0.1, 0.25]

bad_dgs = []
exploding_dgs = []

for data_generator in BENCHMARK_MODELS:

    train_stds = None

    for frac, folder in zip(sigma_frac, sub_folders):

        dg = data_generator()

        if frac is not None:

            dg.model_params["sigma"] = frac * train_stds

            # Planted tank is more sensitive to noise so reduce noise.
            if "PlantedTank" in type(dg).__name__:
                dg.model_params["sigma"] = frac * train_stds * 1e-3

            # Rossler and MooreSpiegle are sensitive to noise so reduce noise.
            if ("Rossler" in type(dg).__name__) or (
                "Moore" in type(dg).__name__):
                dg.model_params["sigma"] = frac * train_stds * 1e-1

        data = dg.generate_data(
            TRAIN_OBS, FORECAST_OBS, NUM_BURN_IN_STATES)
        
        if frac is None:
            train_stds = np.std(data.train_states, axis=0)
        
        description = make_description(dg)

        fcast_mean_error, resp_mean_error = plot_helper(
            folder + type(dg).__name__, data, target_idx=dg.target_idx)

        # Save
        data.to_json(
            DATA_DIR + folder + data_generator.__name__ + ".json",
            model_description=description
        )

        if (resp_mean_error / fcast_mean_error < 1.1) and (
            dg.model_params["sigma"] is None):
            bad_dgs.append(data_generator)

        all_states = np.vstack([
            data.train_states, data.forecast_states, data.interv_states
        ])
        if np.any(np.isnan(all_states)) or np.any(all_states > 1e8):
            exploding_dgs.append(
                {"dg_type": data_generator, "sigma": dg.model_params["sigma"]}
            )

In [ ]:
print(bad_dgs)
for data_generator in bad_dgs:

    print(f"Optimizing {data_generator.__name__}")

    dg = data_generator()
    bad_data = dg.generate_data(
        TRAIN_OBS, FORECAST_OBS, NUM_BURN_IN_STATES)
    
    n = bad_data.train_states.shape[1]
    col_max = np.max(bad_data.train_states, axis=0)
    col_min = np.min(bad_data.train_states, axis=0) 

    # Scale up max and min values:
    col_max = col_max + np.abs(col_max) * 0.5
    col_min = col_min - np.abs(col_min) + 0.5
    
    interv_choices = list(np.arange(n))

    if issubclass(dg.do_intervention_type, interfere.SignalIntervention):
        assert (len(dg.do_intervention_params["intervened_idxs"]) - 1) == len(dg.obs_intervention_params["intervened_idxs"])

        set_diff = set(
            dg.do_intervention_params["intervened_idxs"]).difference(
                dg.obs_intervention_params["intervened_idxs"]
        )
        old_interv_idx = list(set_diff)[0]

        set_inter = set(
            dg.do_intervention_params["intervened_idxs"]).intersection(
                dg.obs_intervention_params["intervened_idxs"]
        )
        for x in set_inter:
            interv_choices.remove(x)

        old_interv_idx_loc = [
            i for i, x in enumerate(dg.do_intervention_params["intervened_idxs"])
            if x == old_interv_idx
        ][0]

    for j in range(300):

        interv_idx = np.random.choice(interv_choices)
        target_choices = list(np.arange(n))
        target_choices.remove(interv_idx)
        target_idx = np.random.choice(target_choices)
        const = col_min[interv_idx] + np.random.rand() * (
            col_max[interv_idx] - col_min[interv_idx])

        # Find an intervention that works.
        if issubclass(dg.do_intervention_type, interfere.SignalIntervention):
            dg.do_intervention_params["intervened_idxs"][old_interv_idx_loc] = interv_idx
            dg.do_intervention_params["signals"][old_interv_idx_loc] = lambda t: const
        else:
            dg.do_intervention_params["intervened_idxs"] = [interv_idx]
            dg.do_intervention_params["constants"] = [const]

        data = dg.generate_data(
        200, 100, 50)

        
        metric = interfere.metrics.rmse
        resp_target = data.interv_states[:, target_idx]
        fcast_target = data.forecast_states[:, target_idx]
        mu_pred = np.mean(data.train_states[:, target_idx])
        avg_pred =  mu_pred * np.ones_like(resp_target)

        fcast_mean_error = metric(fcast_target, avg_pred)
        resp_mean_error = metric(avg_pred, resp_target)

        print(
            f"Attempt {j} -- Error Ratio {resp_mean_error / fcast_mean_error}"
            f"\n\tinterv_index={interv_idx} const={const} target_idx={target_idx}"
            
        )

        if (resp_mean_error / fcast_mean_error > 1.2) and (np.max(np.abs(data.interv_states)) < 1000):
            plot_helper(type(dg).__name__, data, target_idx)
            print(
                f"{type(dg).__name__}"
                f"\nIntervention index: {interv_idx}"
                f"\nInterv constant: {const}"
                f"\ntarget_idx = {target_idx}"
                f"\n\n do_intervention_params = {dg.do_intervention_params}"
            )

            break

    print(f"No Match found for {data_generator.__name__}")